# =============================================================================
# FACIAL EMOTION RECOGNITION - EXPERIMENT TEMPLATE
# =============================================================================
# 
# Universal template for running experiments with different configurations
# Change parameters in Cell 1 to run different experiments
# 
# Author: Pavlo Borysov
# Date: 12/10/2025
# =============================================================================


In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION - CHANGE ONLY HERE!
# =============================================================================
import numpy as np
# Experiment identification
EXPERIMENT_NAME = "test_experiment"  # Will be used in model names and logs
EXPERIMENT_DESCRIPTION = "Testing the template"  # For documentation

# Model architecture
MODEL_TYPE = "rgb"  # "rgb" or "grayscale"
ARCHITECTURE_TYPE = "complex"  # "simple" or "complex"

# Architecture parameters
if ARCHITECTURE_TYPE == "complex":
    CHANNELS = (64, 128, 512, 512, 128)
    CONVS_PER_BLOCK = 2
else:
    CHANNELS = (32, 64, 128)
    CONVS_PER_BLOCK = 1

# Regularization parameters
DROPOUT_HEAD = 0.3
L2_REG = 0.001

# Loss
# Fixed class order and weights (from main notebook)
CLASS_ORDER = ['happy', 'neutral', 'sad', 'surprise']
CLASS_WEIGHTS = {
    0: 0.950,  # happy
    1: 0.950,  # neutral  
    2: 0.949,  # sad
    3: 1.190   # surprise
}

alpha_raw = np.array([CLASS_WEIGHTS[i] for i in range(len(CLASS_ORDER))], dtype=np.float32)
alpha_mean = alpha_raw.mean()
alpha_vec = alpha_raw / alpha_mean

LOSS = "sparse_categorical_crossentropy"
LABEL_SMOOTHING = 0.05
USE_CLASS_WEIGHT = True

# Training parameters
LEARNING_RATE = 0.0001
EPOCHS = 50
BATCH_SIZE = 64
PATIENCE = 10  # Early stopping patience

# Data parameters
IMG_SIZE = (48, 48)
COLOR_MODE = "rgb" if MODEL_TYPE == "rgb" else "grayscale"
AUGMENT = True

# Auto-generated names
MODEL_NAME = f"{ARCHITECTURE_TYPE}_cnn_{MODEL_TYPE}_{EXPERIMENT_NAME}"
RUN_DIR_NAME = f"{MODEL_TYPE}_{EXPERIMENT_NAME}"

# Display configuration
print("=" * 80)
print(f"EXPERIMENT: {EXPERIMENT_NAME}")
print("=" * 80)
print(f"Description: {EXPERIMENT_DESCRIPTION}")
print(f"Model type: {MODEL_TYPE}")
print(f"Architecture: {ARCHITECTURE_TYPE} - {CHANNELS}")
print(f"Image size: {IMG_SIZE}")
print(f"Color mode: {COLOR_MODE}")
print(f"Dropout: {DROPOUT_HEAD}")
print(f"L2 reg: {L2_REG}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Augmentation: {AUGMENT}")
print(f"Model name: {MODEL_NAME}")
print("=" * 80)


In [ ]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

import warnings
import os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import time
import json
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight 

import tensorflow as tf
import keras
from keras import layers
from keras.utils import image_dataset_from_directory

# Fixed random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"Random seed: {RANDOM_SEED}")


In [ ]:
# =============================================================================
# CONSTANTS AND PATHS
# =============================================================================

# Data paths
DATA_DIR = Path("../../data")
TRAIN_DATA_DIR = Path("../../data/train")
VALIDATION_DATA_DIR = Path("../../data/validation")
TEST_DATA_DIR = Path("../../data/test")

# Output paths
MODELS_DIR = Path("../../models")
RUNS_DIR = Path("../../runs")
REPORTS_DIR = Path("../../reports")
SUMMARY_CSV = REPORTS_DIR / "models_summary.csv"

# Model parameters
NUM_CLASSES = 4
METRIC_NAME = "sparse_categorical_accuracy"
VAL_METRIC_NAME = f"val_{METRIC_NAME}"

print("Setup complete!")
print(f"Class order: {CLASS_ORDER}")
print(f"Class weights: {CLASS_WEIGHTS}")


In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def save_metrics(metrics: dict, out_json: Path):
    """Save metrics to JSON file."""
    with open(out_json, "w") as f:
        json.dump(metrics, f, indent=2)

def make_run_dir(name: str) -> Path:
    """Create unique directory for experiment."""
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = RUNS_DIR / f"{ts}_{name}"
    (run_dir / "figs").mkdir(parents=True, exist_ok=True)
    return run_dir

def print_dict_table(d: dict, title="Dictionary", val_header="Value", show_percent=False, value_decimals=3):
    """Print dictionary as formatted table."""
    print(f"\n{title}")
    print(f"{'Class':<8} {val_header}")
    print("-" * (8 + len(val_header) + 2))
    total = sum(d.values()) if show_percent else None
    
    for key, value in d.items():
        if show_percent and total:
            pct = f" ({value/total*100:.1f}%)"
            print(f"{key:<8} {value:.{value_decimals}f}{pct}")
        else:
            print(f"{key:<8} {value:.{value_decimals}f}")
    
    if show_percent and total:
        print("-" * (8 + len(val_header) + 2))
        print(f"{'TOTAL':<8} {total:.{value_decimals}f}")
    print()

print("Utility functions loaded!")


In [ ]:
# =============================================================================
# DATA LOADERS
# =============================================================================

def make_generators(train_dir, val_dir, test_dir, img_size=(48,48),
                    color_mode="grayscale", batch_size=64, augment=True):
    """Create data generators with fixed class order."""
    
    def preprocess_image(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    
    def augment_image_simple(image, label):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.2)
        image = tf.image.random_contrast(image, 0.8, 1.2)
        image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label
    
    print("Creating data loaders...")
    
    # Train data
    train_ds = image_dataset_from_directory(
        train_dir,
        class_names=CLASS_ORDER,
        image_size=img_size,
        batch_size=batch_size,
        color_mode=color_mode,
        shuffle=True
    )
    train_ds = train_ds.map(preprocess_image)
    if augment:
        train_ds = train_ds.map(augment_image_simple)
    
    # Validation data
    val_ds = image_dataset_from_directory(
        val_dir,
        class_names=CLASS_ORDER,
        image_size=img_size,
        batch_size=batch_size,
        color_mode=color_mode,
        shuffle=False
    )
    val_ds = val_ds.map(preprocess_image)
    
    # Test data
    test_ds = image_dataset_from_directory(
        test_dir,
        class_names=CLASS_ORDER,
        image_size=img_size,
        batch_size=batch_size,
        color_mode=color_mode,
        shuffle=False
    )
    test_ds = test_ds.map(preprocess_image)
    
    return train_ds, val_ds, test_ds

# Create data loaders
train_ds, val_ds, test_ds = make_generators(
    train_dir=TRAIN_DATA_DIR,
    val_dir=VALIDATION_DATA_DIR,
    test_dir=TEST_DATA_DIR,
    img_size=IMG_SIZE,
    color_mode=COLOR_MODE,
    batch_size=BATCH_SIZE,
    augment=AUGMENT
)

# Data sanity check
for xb, yb in train_ds.take(1):
    break

print(f"\nData loaded successfully!")
print(f"Train batch shape: {xb.shape}")
print(f"Labels shape: {yb.shape}")
print(f"Image range: [{tf.reduce_min(xb):.3f}, {tf.reduce_max(xb):.3f}]")
print(f"Unique labels: {tf.unique(yb)[0].numpy()}")
print(f"Augmentation: {AUGMENT}")


In [ ]:
def sparse_focal_loss(gamma=2.0, alpha=None, eps=1e-7):
    """
    Focal loss for sparse labels with softmax probabilities.
    alpha:
      - None -> no class weighting
      - list/tuple/np.array of shape [num_classes]
      - tf.Tensor same shape
    """
    alpha = None if alpha is None else tf.convert_to_tensor(alpha, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        # y_true: (batch,), ints; y_pred: (batch, num_classes), softmax probs
        y_true = tf.cast(y_true, tf.int32)
        num_classes = tf.shape(y_pred)[-1]
        y_true_oh = tf.one_hot(y_true, depth=num_classes, dtype=tf.float32)

        # pt = prob. of the true class
        pt = tf.reduce_sum(y_true_oh * tf.clip_by_value(y_pred, eps, 1.0), axis=-1)

        # alpha weighting per class (if provided)
        if alpha is not None:
            # gather alpha for the true class
            alpha_t = tf.reduce_sum(y_true_oh * alpha, axis=-1)
        else:
            alpha_t = 1.0

        focal = - alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        return tf.reduce_mean(focal)

    return loss_fn

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE
# =============================================================================

def build_cnn(input_shape, num_classes, channels=(64, 128, 256), 
               convs_per_block=2, dropout_head=0.3, activation="relu",
               l2_reg=0.001, lr=0.001, loss="sparse_categorical_crossentropy",
               label_smoothing=0.0):
    """Build CNN model with configurable architecture."""
    
    # Input layer
    inputs = keras.Input(shape=input_shape, name="input")
    x = inputs
    
    # Convolutional blocks
    for i, filters in enumerate(channels):
        for j in range(convs_per_block):
            x = layers.Conv2D(
                filters, 3, padding="same",
                kernel_regularizer=keras.regularizers.l2(l2_reg),
                name=f"conv_{i+1}_{j+1}"
            )(x)
            x = layers.BatchNormalization(name=f"bn_{i+1}_{j+1}")(x)
            
            if activation == "leakyrelu":
                x = layers.LeakyReLU(name=f"leaky_re_lu_{i+1}_{j+1}" if i==0 and j==0 else f"leaky_re_lu_{i+1}_{j+1}")(x)
            else:
                x = layers.Activation("relu", name=f"activation_{i+1}_{j+1}")(x)
        
        # Max pooling after each block (except last)
        if i < len(channels) - 1:
            x = layers.MaxPooling2D(2, name=f"pool_{i+1}")(x)
    
    # Global average pooling
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    
    # Dropout
    x = layers.Dropout(dropout_head, name="dropout_head")(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)
    
    # Create model
    model = keras.Model(inputs, outputs, name="emotion_cnn")

    # configure loss
    if callable(LOSS):
        loss_fn = LOSS
    elif LOSS == "sparse_categorical_crossentropy":
        loss_fn = keras.losses.SparseCategoricalCrossentropy(
            label_smoothing=LABEL_SMOOTHING
        )
    else:
        loss_fn = LOSS  # e.g. built-in string losses    
    
    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=loss_fn,
        metrics=[METRIC_NAME]
    )
    
    return model

# Create model
input_shape = (*IMG_SIZE, 3 if MODEL_TYPE == "rgb" else 1)
model = build_cnn(
    input_shape=input_shape,
    num_classes=NUM_CLASSES,
    channels=CHANNELS,
    convs_per_block=CONVS_PER_BLOCK,
    dropout_head=DROPOUT_HEAD,
    activation="leakyrelu",
    l2_reg=L2_REG,
    lr=LEARNING_RATE
)

print(f"\nModel created: {MODEL_NAME}")
print(f"Input shape: {input_shape}")
print(f"Parameters: {model.count_params():,}")
model.summary()


In [ ]:
# =============================================================================
# TRAINING
# =============================================================================

# Create callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / f"best_{MODEL_NAME}.keras"),
        monitor=VAL_METRIC_NAME,
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Create run directory
run_dir = make_run_dir(RUN_DIR_NAME)

# Training configuration
print("=" * 70)
print(f"TRAINING CONFIGURATION - {MODEL_NAME}")
print("=" * 70)
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Dropout: {DROPOUT_HEAD}")
print(f"L2 reg: {L2_REG}")
print(f"Class weights: {CLASS_WEIGHTS}")
print("=" * 70)

# Start training
start_time = time.time()
start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"\nTraining started at: {start_str}")

fit_class_weight = CLASS_WEIGHTS if USE_CLASS_WEIGHT else None
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=fit_class_weight,
    verbose=1
)

end_time = time.time()
training_time = end_time - start_time

print(f"\nTraining completed in {training_time/60:.1f} minutes")


In [ ]:
# =============================================================================
# EVALUATION
# =============================================================================

print("=" * 70)
print(f"EVALUATION - {MODEL_NAME}")
print("=" * 70)

# Get predictions
y_true = []
y_pred = []

for batch in test_ds:
    images, labels = batch
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

# Calculate metrics
test_accuracy = f1_score(y_true, y_pred, average='weighted')
test_macro_f1 = f1_score(y_true, y_pred, average='macro')
test_weighted_f1 = f1_score(y_true, y_pred, average='weighted')

# Display results
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test macro F1: {test_macro_f1:.3f}")
print(f"Test weighted F1: {test_weighted_f1:.3f}")
print("=" * 70)

# Save results
results = {
    "experiment_name": EXPERIMENT_NAME,
    "experiment_description": EXPERIMENT_DESCRIPTION,
    "model_name": MODEL_NAME,
    "model_type": MODEL_TYPE,
    "architecture_type": ARCHITECTURE_TYPE,
    "channels": CHANNELS,
    "convs_per_block": CONVS_PER_BLOCK,
    "input_shape": list(input_shape),
    "dropout_head": DROPOUT_HEAD,
    "l2_reg": L2_REG,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "augment": AUGMENT,
    "training_time_minutes": training_time / 60,
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_weighted_f1": test_weighted_f1,
    "total_epochs_trained": len(history.history['loss']),
    "final_train_acc": history.history[METRIC_NAME][-1],
    "final_train_loss": history.history['loss'][-1],
    "final_val_acc": history.history[VAL_METRIC_NAME][-1],
    "final_val_loss": history.history['val_loss'][-1],
    "best_val_acc": max(history.history[VAL_METRIC_NAME]),
    "best_epoch": history.history[VAL_METRIC_NAME].index(max(history.history[VAL_METRIC_NAME])) + 1
}

# Save to run directory
save_metrics(results, run_dir / "experiment_results.json")
save_metrics(results, run_dir / "config.json")

print(f"\nResults saved to: {run_dir}")
print(f"Experiment completed: {EXPERIMENT_NAME}")


In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy plot
axes[0].plot(history.history[METRIC_NAME], label='train', linewidth=2)
axes[0].plot(history.history[VAL_METRIC_NAME], label='validation', linewidth=2)
axes[0].set_title(f'Accuracy - {MODEL_NAME}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='validation', linewidth=2)
axes[1].set_title(f'Loss - {MODEL_NAME}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar()

tick_marks = np.arange(len(CLASS_ORDER))
plt.xticks(tick_marks, CLASS_ORDER, rotation=45)
plt.yticks(tick_marks, CLASS_ORDER)

# Add text annotations
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, cm[i, j], horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black", fontsize=10)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.tight_layout()
plt.savefig(run_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Visualizations saved to: {run_dir}")
